# 01 — Exploratory Data Analysis

**Dataset**: Yakimov et al. (2024) — Fingernail bed images with hemoglobin (Hb) labels.

## Objectives
1. Load and inspect the dataset structure
2. Analyze Hb value distribution (g/L and g/dL)
3. Map WHO anemia severity classes
4. Assess image quality and resolution consistency
5. Visualize sample fingernail images with nail & skin bounding boxes across Hb ranges

In [ ]:
import os
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from pathlib import Path
from PIL import Image
from collections import Counter

sns.set_theme(style="whitegrid", palette="muted")
%matplotlib inline

# Configuration
DATA_ROOT = Path("../data/raw")
METADATA_CSV = DATA_ROOT / "metadata.csv"
PHOTO_DIR = DATA_ROOT / "photo"

print(f"Data root : {DATA_ROOT.resolve()}")
print(f"CSV exists: {METADATA_CSV.exists()}")
print(f"Photo dir : {PHOTO_DIR.exists()}")
print(f"Num photos: {len(list(PHOTO_DIR.glob('*.jpg')))}")

## 1. Load Metadata

In [ ]:
# Load the metadata CSV
df = pd.read_csv(METADATA_CSV)

# Parse bounding box columns from string to list
df["NAIL_BOUNDING_BOXES"] = df["NAIL_BOUNDING_BOXES"].apply(ast.literal_eval)
df["SKIN_BOUNDING_BOXES"] = df["SKIN_BOUNDING_BOXES"].apply(ast.literal_eval)

# Derived columns
df["hb_gdL"] = df["HB_LEVEL_GperL"] / 10.0  # Convert g/L → g/dL
df["image_path"] = df["PATIENT_ID"].apply(lambda pid: PHOTO_DIR / f"{pid}.jpg")
df["image_exists"] = df["image_path"].apply(lambda p: p.exists())
df["num_nail_boxes"] = df["NAIL_BOUNDING_BOXES"].apply(len)
df["num_skin_boxes"] = df["SKIN_BOUNDING_BOXES"].apply(len)

# WHO anemia classification (using g/dL thresholds — gender-agnostic simplified)
def classify_anemia(hb_gdL):
    if hb_gdL < 7.0:
        return "Severe"
    elif hb_gdL < 10.0:
        return "Moderate"
    elif hb_gdL < 12.0:
        return "Mild"
    else:
        return "Normal"

df["anemia_class"] = df["hb_gdL"].apply(classify_anemia)

print(f"Loaded {len(df)} samples")
print(f"Images found: {df['image_exists'].sum()} / {len(df)}")
print(f"Unique measurement dates (sessions): {df['MEASUREMENT_DATE'].nunique()}")
print()
display(df[["PATIENT_ID", "HB_LEVEL_GperL", "hb_gdL", "anemia_class", "num_nail_boxes", "image_exists"]].head(10))
print()
display(df.describe())

## 2. Hemoglobin Distribution & Anemia Classification

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) Histogram of Hb in g/dL
axes[0].hist(df["hb_gdL"], bins=25, edgecolor="black", alpha=0.7, color="steelblue")
axes[0].axvline(x=7.0, color="red", linestyle="--", lw=1.5, label="Severe (<7)")
axes[0].axvline(x=10.0, color="orange", linestyle="--", lw=1.5, label="Moderate (<10)")
axes[0].axvline(x=12.0, color="gold", linestyle="--", lw=1.5, label="Mild (<12)")
axes[0].set_xlabel("Hemoglobin (g/dL)")
axes[0].set_ylabel("Count")
axes[0].set_title("Hb Distribution")
axes[0].legend(fontsize=9)

# (b) Box plot
sns.boxplot(y=df["hb_gdL"], ax=axes[1], color="lightcoral")
axes[1].set_ylabel("Hemoglobin (g/dL)")
axes[1].set_title("Hb Box Plot")

# (c) Anemia class distribution
class_order = ["Severe", "Moderate", "Mild", "Normal"]
class_colors = ["#d62728", "#ff7f0e", "#ffbb33", "#2ca02c"]
counts = df["anemia_class"].value_counts().reindex(class_order, fill_value=0)
axes[2].bar(counts.index, counts.values, color=class_colors, edgecolor="black")
axes[2].set_ylabel("Count")
axes[2].set_title("WHO Anemia Classes")
for i, v in enumerate(counts.values):
    axes[2].text(i, v + 1, str(v), ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

# Key stats
print(f"\n{'='*40}")
print(f"Mean Hb:   {df['hb_gdL'].mean():.2f} g/dL  ({df['HB_LEVEL_GperL'].mean():.1f} g/L)")
print(f"Median Hb: {df['hb_gdL'].median():.2f} g/dL")
print(f"Std Hb:    {df['hb_gdL'].std():.2f} g/dL")
print(f"Range:     [{df['hb_gdL'].min():.1f}, {df['hb_gdL'].max():.1f}] g/dL")
print(f"\nClass breakdown:")
for cls in class_order:
    n = (df["anemia_class"] == cls).sum()
    pct = 100 * n / len(df)
    print(f"  {cls:10s}: {n:3d}  ({pct:.1f}%)")

## 3. Session (Measurement Date) Analysis

In [ ]:
# Analyze how samples are distributed across measurement sessions
session_stats = df.groupby("MEASUREMENT_DATE").agg(
    n_samples=("PATIENT_ID", "count"),
    mean_hb=("hb_gdL", "mean"),
    std_hb=("hb_gdL", "std"),
    min_hb=("hb_gdL", "min"),
    max_hb=("hb_gdL", "max"),
).reset_index()

print(f"Number of sessions: {len(session_stats)}")
print(f"Samples per session: {session_stats['n_samples'].min()} – {session_stats['n_samples'].max()} "
      f"(mean {session_stats['n_samples'].mean():.1f})")
print()
display(session_stats)

## 4. Image Quality Assessment

In [ ]:
def get_image_stats(image_path: Path) -> dict:
    """Compute basic image statistics."""
    img = Image.open(image_path)
    arr = np.array(img)
    return {
        "width": img.width,
        "height": img.height,
        "channels": arr.shape[2] if arr.ndim == 3 else 1,
        "mean_brightness": arr.mean(),
        "std_brightness": arr.std(),
    }

# Profile all available images
valid_df = df[df["image_exists"]].copy()
stats = []
for _, row in valid_df.iterrows():
    stats.append(get_image_stats(row["image_path"]))

stats_df = pd.DataFrame(stats, index=valid_df.index)

print(f"Profiled {len(stats_df)} images:\n")
display(stats_df.describe())

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].hist(stats_df["width"], bins=20, edgecolor="black", alpha=0.7)
axes[0].set_title("Image Width Distribution")
axes[0].set_xlabel("Width (px)")

axes[1].hist(stats_df["height"], bins=20, edgecolor="black", alpha=0.7, color="orange")
axes[1].set_title("Image Height Distribution")
axes[1].set_xlabel("Height (px)")

axes[2].scatter(stats_df["mean_brightness"], valid_df["hb_gdL"].values, alpha=0.5, s=15)
axes[2].set_xlabel("Mean Brightness")
axes[2].set_ylabel("Hb (g/dL)")
axes[2].set_title("Brightness vs Hb")

plt.tight_layout()
plt.show()

## 5. Sample Visualization with Bounding Boxes

In [ ]:
def show_sample_with_boxes(row, ax):
    """Display an image with nail (green) and skin (blue) bounding boxes."""
    img = Image.open(row["image_path"])
    ax.imshow(img)
    
    # Draw nail bounding boxes (green)
    for box in row["NAIL_BOUNDING_BOXES"]:
        x1, y1, x2, y2 = box
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                  linewidth=2, edgecolor="lime", facecolor="none")
        ax.add_patch(rect)
    
    # Draw skin bounding boxes (blue)
    for box in row["SKIN_BOUNDING_BOXES"]:
        x1, y1, x2, y2 = box
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                  linewidth=2, edgecolor="cyan", facecolor="none", linestyle="--")
        ax.add_patch(rect)
    
    ax.set_title(f"ID {row['PATIENT_ID']} — Hb={row['hb_gdL']:.1f} g/dL ({row['anemia_class']})",
                 fontsize=10)
    ax.axis("off")

# Show 2 samples per anemia class
class_order = ["Severe", "Moderate", "Mild", "Normal"]
fig, axes = plt.subplots(4, 2, figsize=(14, 20))

for i, cls in enumerate(class_order):
    subset = valid_df[valid_df["anemia_class"] == cls]
    samples = subset.sample(min(2, len(subset)), random_state=42)
    for j, (_, row) in enumerate(samples.iterrows()):
        show_sample_with_boxes(row, axes[i][j])
    # Blank unused
    for j in range(len(samples), 2):
        axes[i][j].axis("off")
        axes[i][j].set_title(f"({cls} — no more samples)")

fig.suptitle("Samples by Anemia Class — Nail (green) & Skin (cyan) ROIs", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print("Legend: GREEN = nail bounding box, CYAN = skin bounding box")

## 6. Summary & Key Takeaways

**Dataset Overview:**
- 251 samples across multiple measurement sessions, each with 3 nail/skin bounding boxes
- Hb values span 4.4–16.9 g/dL — covering severe anemia through healthy normals
- Class imbalance: most samples cluster in Normal/Mild — very few Severe cases

**Implications for Modeling:**
- Regression target: `HB_LEVEL_GperL` (or derived `hb_gdL`)
- ROI strategy: use **provided bounding boxes** to crop nail and skin patches (no auto-detection needed)
- Need to handle class imbalance: weighted loss or oversampling for severe/moderate cases
- Session-level splits recommended to avoid data leakage (same session = same lighting conditions)

**Next Steps:**
- Proceed to `02_feature_extraction.ipynb` for ROI cropping and color-space analysis
- Check for class imbalance and consider stratified splits in `03_baseline_models.ipynb`